# LTX-Video 2 — Colab Pro Server
Run this notebook on Colab Pro (A100/T4). It starts a video generation server and exposes it via ngrok.
Copy the ngrok URL into your `.env` as `COLAB_URL`.

In [ ]:
# ── CELL 1: Install dependencies ──
!pip install -q diffusers transformers accelerate sentencepiece pyngrok flask
!pip install -q 'huggingface_hub[hf_transfer]'
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print('✅ Dependencies installed')

In [ ]:
# ── CELL 2: Load LTX-Video 2 model ──
import torch
from diffusers import LTXPipeline

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

pipe = LTXPipeline.from_pretrained(
    'Lightricks/LTX-Video',
    torch_dtype=torch.bfloat16
)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
print('✅ LTX-Video model loaded')

In [ ]:
# ── CELL 3: Define generation function ──
import tempfile, base64
from diffusers.utils import export_to_video

def generate_video(prompt: str, num_frames: int = 97, fps: int = 24, height: int = 480, width: int = 704) -> str:
    """
    Generate a video and return as base64 MP4 string.
    num_frames must satisfy: num_frames % 8 == 1  (e.g. 25, 49, 73, 97)
    """
    print(f'Generating: {prompt[:60]}...')
    output = pipe(
        prompt=prompt,
        negative_prompt='worst quality, inconsistent motion, blurry, jittery, distorted',
        width=width,
        height=height,
        num_frames=num_frames,
        num_inference_steps=40,
        guidance_scale=3.0,
    )
    frames = output.frames[0]
    tmp = tempfile.NamedTemporaryFile(suffix='.mp4', delete=False)
    export_to_video(frames, tmp.name, fps=fps)
    with open(tmp.name, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode('utf-8')
    print(f'✅ Done: {len(b64)//1024} KB')
    return b64

print('✅ Generator ready')

In [ ]:
# ── CELL 4: Start Flask server + ngrok ──
# Paste your ngrok authtoken from https://dashboard.ngrok.com/authtokens
NGROK_TOKEN = 'YOUR_NGROK_AUTHTOKEN_HERE'

from flask import Flask, request, jsonify
from pyngrok import ngrok, conf
import threading

conf.get_default().auth_token = NGROK_TOKEN

app = Flask(__name__)

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'model': 'LTX-Video-2'})

@app.route('/generate', methods=['POST'])
def generate():
    data = request.get_json()
    prompt = data.get('prompt', '')
    if not prompt:
        return jsonify({'error': 'prompt required'}), 400
    try:
        b64 = generate_video(
            prompt=prompt,
            num_frames=data.get('num_frames', 97),
            fps=data.get('fps', 24),
            height=data.get('height', 480),
            width=data.get('width', 704),
        )
        return jsonify({'success': True, 'video_base64': b64})
    except Exception as e:
        return jsonify({'success': False, 'error': str(e)}), 500

# Start Flask in background thread
thread = threading.Thread(target=lambda: app.run(port=5000, use_reloader=False))
thread.daemon = True
thread.start()

# Open ngrok tunnel
public_url = ngrok.connect(5000).public_url
print('=' * 60)
print(f'✅ Server running!')
print(f'   COLAB_URL = {public_url}')
print('=' * 60)
print('Copy the URL above into your .env as COLAB_URL')